# 🚗 Automotive Data Explorer → Static `index.html` Builder

This notebook turns your Excel dataset into a **single static `index.html`** that works on **GitHub Pages** (no Gradio, no server, no Python needed to view it).

**How to use**
1. Run cell 1 (installs).
2. Run cell 2 and **upload your Excel file** when prompted (e.g. `2020-2026_Automotive_Groups_Horsepower_MSRP_2026_completed.xlsx`).
3. Run cells 3 and 4 – the last cell downloads `index.html`.
4. Put `index.html` in the root of a GitHub repo → **Settings → Pages → Deploy from branch (main / root)**.

The page includes: live search, X/Y axis scatter plot, MSRP box plot by parent company, price summary by model year, price-change-by-company (when more than one year exists), and a top-50 table preview.

### 1. Install dependencies

In [ ]:
!pip install -q pandas openpyxl

### 2. Upload your Excel file

In [ ]:
import os

# Leave blank to get an upload prompt.
# Or, if the file is already in the Colab file panel, set e.g. FILE_PATH = "/content/yourfile.xlsx"
FILE_PATH = ""

if not FILE_PATH:
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file was uploaded. Run this cell again and choose your Excel file.")
    FILE_PATH = os.path.abspath(list(uploaded.keys())[0])

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"'{FILE_PATH}' not found. Upload it via the Files panel (folder icon, left sidebar) "
        "or leave FILE_PATH blank and re-run to use the upload prompt."
    )

print("Using file:", FILE_PATH)

### 3. Load and prepare the data

In [ ]:
import re
import json
import pandas as pd
from IPython.display import display

if FILE_PATH.lower().endswith((".xlsx", ".xlsm", ".xls")):
    df = pd.read_excel(FILE_PATH)
elif FILE_PATH.lower().endswith(".csv"):
    df = pd.read_csv(FILE_PATH)
else:
    raise ValueError("Please provide an .xlsx, .xls or .csv file.")

df.columns = [str(c).strip() for c in df.columns]
df = df.dropna(how="all")

def find_col(preferred, keywords):
    if preferred in df.columns:
        return preferred
    for c in df.columns:
        if any(k in c.lower() for k in keywords):
            return c
    return None

MSRP_COL  = find_col("Starting MSRP (USD)",       ["msrp", "price"])
YEAR_COL  = find_col("Model Year",                ["year"])
GROUP_COL = find_col("Parent Company / Group",    ["parent", "group", "manufacturer", "make"])
HP_COL    = find_col("Horsepower",                ["horsepower", "hp"])

def to_number(series):
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)
    return pd.to_numeric(cleaned.replace("", pd.NA), errors="coerce")

if MSRP_COL:
    df[MSRP_COL] = to_number(df[MSRP_COL])
if HP_COL:
    df[HP_COL] = to_number(df[HP_COL])
if YEAR_COL:
    df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce").astype("Int64")

# Make everything JSON-friendly (dates -> strings)
for c in df.columns:
    if pd.api.types.is_datetime64_any_dtype(df[c]):
        df[c] = df[c].dt.strftime("%Y-%m-%d")

print(f"Rows: {len(df):,}   Columns: {len(df.columns)}")
print("Detected -> MSRP:", MSRP_COL, "| Year:", YEAR_COL, "| Group:", GROUP_COL, "| Horsepower:", HP_COL)
display(df.head())

### 4. Build `index.html` and download it

In [ ]:
from google.colab import files

TITLE = "Automotive Data Explorer"   # <- change the page title here
OUTPUT_FILE = "index.html"

HTML_TEMPLATE = r"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>__TITLE__</title>
<script src="https://cdn.plot.ly/plotly-2.35.2.min.js" charset="utf-8"></script>
<style>
  :root { --bg:#f6f7fb; --card:#fff; --text:#1f2430; --muted:#6b7280; --accent:#2563eb; --border:#e5e7eb; }
  * { box-sizing: border-box; }
  body { margin:0; font-family: system-ui, -apple-system, "Segoe UI", Roboto, sans-serif; background:var(--bg); color:var(--text); }
  header { padding: 28px 20px 8px; max-width: 1200px; margin: 0 auto; }
  header h1 { margin:0 0 6px; font-size: 1.8rem; }
  header p { margin:0; color:var(--muted); }
  main { max-width:1200px; margin:0 auto; padding: 12px 20px 48px; }
  section { background:var(--card); border:1px solid var(--border); border-radius:12px; padding:16px 18px; margin-top:16px; }
  section h2 { margin:0 0 12px; font-size:1.1rem; }
  .controls { display:grid; grid-template-columns: 2fr 1fr 1fr auto; gap:12px; align-items:end; }
  @media (max-width:760px){ .controls { grid-template-columns:1fr; } }
  label { display:flex; flex-direction:column; font-size:.8rem; color:var(--muted); gap:4px; }
  input, select { padding:9px 10px; border:1px solid var(--border); border-radius:8px; font-size:.95rem; background:#fff; color:var(--text); }
  button { padding:10px 16px; border:0; border-radius:8px; background:var(--accent); color:#fff; font-weight:600; cursor:pointer; }
  button:hover { filter:brightness(1.08); }
  #count { margin:10px 2px 0; color:var(--muted); font-size:.9rem; }
  .tablewrap { overflow:auto; max-height:520px; border:1px solid var(--border); border-radius:8px; }
  table { border-collapse:collapse; width:100%; font-size:.85rem; }
  th, td { padding:7px 10px; border-bottom:1px solid var(--border); text-align:left; white-space:nowrap; }
  th { position:sticky; top:0; background:#f3f4f6; z-index:1; }
  tr:hover td { background:#f9fafb; }
  td.num, th.num { text-align:right; }
  .note { color:var(--muted); font-size:.9rem; margin:0; }
  .hidden { display:none; }
</style>
</head>
<body>
<header>
  <h1>🚗 __TITLE__</h1>
  <p>Search the dataset, pick axes, and explore the charts. Everything runs in your browser.</p>
</header>
<main>
  <section>
    <div class="controls">
      <label>Search anything (e.g. BMW, Toyota, 2024)
        <input id="search" type="search" placeholder="Type to filter the table and charts...">
      </label>
      <label>X axis <select id="xsel"></select></label>
      <label>Y axis <select id="ysel"></select></label>
      <button id="reset" type="button">Reset</button>
    </div>
    <p id="count"></p>
  </section>

  <section>
    <h2>Interactive visualization</h2>
    <div id="scatter" style="width:100%;height:480px;"></div>
  </section>

  <section id="boxsec">
    <h2 id="boxtitle">Starting MSRP by parent company</h2>
    <div id="box" style="width:100%;"></div>
  </section>

  <section id="summarysec">
    <h2>Price summary by model year</h2>
    <div class="tablewrap" id="summary"></div>
  </section>

  <section id="changesec">
    <h2 id="changetitle">Price change by parent company</h2>
    <p class="note" id="changenote"></p>
    <div id="changechart" style="width:100%;"></div>
    <div class="tablewrap" id="changetable" style="margin-top:12px;"></div>
  </section>

  <section>
    <h2>Dataset preview (top 50 matches)</h2>
    <div class="tablewrap"><table id="tbl"></table></div>
  </section>
</main>

<script>
const DATA = __DATA__;
const CFG = __CONFIG__;
const COLS = CFG.columns;
const $ = id => document.getElementById(id);

const esc = v => String(v).replace(/&/g,"&amp;").replace(/</g,"&lt;").replace(/>/g,"&gt;").replace(/"/g,"&quot;");
const isNum = v => typeof v === "number" && isFinite(v);
const fmtNum = v => v.toLocaleString(undefined, {maximumFractionDigits: 2});
const fmtMoney = v => "$" + v.toLocaleString(undefined, {maximumFractionDigits: 0});
const cell = (v, col) => {
  if (v === null || v === undefined) return "";
  if (isNum(v)) return (col === CFG.year_col) ? String(v) : fmtNum(v);
  return esc(v);
};
const median = a => { const s=[...a].sort((x,y)=>x-y), m=Math.floor(s.length/2); return s.length ? (s.length%2 ? s[m] : (s[m-1]+s[m])/2) : NaN; };
const mean = a => a.reduce((x,y)=>x+y,0)/a.length;

// Precompute a lowercase search string per row
const HAY = DATA.map(r => COLS.map(c => r[c] === null || r[c] === undefined ? "" : String(r[c])).join(" | ").toLowerCase());

function filtered() {
  const q = $("search").value.trim().toLowerCase();
  if (!q) return DATA;
  return DATA.filter((_, i) => HAY[i].includes(q));
}

function fillSelect(sel, def) {
  sel.innerHTML = "";
  COLS.forEach(c => {
    const o = document.createElement("option");
    o.value = c; o.textContent = c; if (c === def) o.selected = true;
    sel.appendChild(o);
  });
}

function drawScatter(rows) {
  const x = $("xsel").value, y = $("ysel").value;
  const hover = rows.map(r => COLS.map(c => esc(c) + ": " + cell(r[c], c)).join("<br>"));
  Plotly.react("scatter", [{
    type: "scatter", mode: "markers",
    x: rows.map(r => r[x]), y: rows.map(r => r[y]),
    text: hover, hoverinfo: "text",
    marker: { size: 9, opacity: 0.75 }
  }], {
    title: { text: y + " vs " + x },
    xaxis: { title: { text: x } }, yaxis: { title: { text: y } },
    template: "plotly_white", margin: { t: 50, l: 70, r: 20, b: 60 }, hovermode: "closest"
  }, { responsive: true });
}

function drawBox(rows) {
  const sec = $("boxsec");
  if (!CFG.group_col || !CFG.msrp_col) { sec.classList.add("hidden"); return; }
  const groups = new Map();
  rows.forEach(r => {
    const g = r[CFG.group_col], v = r[CFG.msrp_col];
    if (g === null || g === undefined || !isNum(v)) return;
    if (!groups.has(g)) groups.set(g, []);
    groups.get(g).push(v);
  });
  if (!groups.size) { Plotly.purge("box"); $("box").innerHTML = "<p class='note'>No data for this search.</p>"; return; }
  const sorted = [...groups.entries()].sort((a, b) => median(a[1]) - median(b[1])); // ascending -> highest median on top
  const traces = sorted.map(([g, vals]) => ({
    type: "box", orientation: "h", name: String(g), x: vals,
    boxpoints: "all", jitter: 0.4, pointpos: 0, marker: { size: 4, opacity: 0.55 }, showlegend: false
  }));
  $("box").innerHTML = "";
  Plotly.react("box", traces, {
    title: { text: "Distribution of " + CFG.msrp_col + " by " + CFG.group_col + (CFG.year_col ? "" : "") },
    xaxis: { title: { text: CFG.msrp_col }, gridcolor: "#e5e7eb" },
    margin: { t: 50, l: 220, r: 20, b: 60 },
    height: Math.max(420, sorted.length * 42 + 140)
  }, { responsive: true });
}

function table(headers, rowsHtml) {
  return "<table><thead><tr>" + headers.map(h => "<th class='" + (h.num ? "num" : "") + "'>" + esc(h.t) + "</th>").join("") +
         "</tr></thead><tbody>" + rowsHtml.join("") + "</tbody></table>";
}

function drawSummary(rows) {
  const sec = $("summarysec");
  if (!CFG.year_col || !CFG.msrp_col) { sec.classList.add("hidden"); return; }
  const byYear = new Map();
  rows.forEach(r => {
    const y = r[CFG.year_col], v = r[CFG.msrp_col];
    if (!isNum(y) || !isNum(v)) return;
    if (!byYear.has(y)) byYear.set(y, []);
    byYear.get(y).push(v);
  });
  const years = [...byYear.keys()].sort((a, b) => a - b);
  if (!years.length) { $("summary").innerHTML = "<p class='note' style='padding:10px'>No data for this search.</p>"; return; }
  const heads = [{t:"Model Year"},{t:"Average MSRP",num:1},{t:"Median MSRP",num:1},{t:"Min MSRP",num:1},{t:"Max MSRP",num:1},{t:"Vehicle Count",num:1}];
  const body = years.map(y => {
    const a = byYear.get(y);
    return "<tr><td>" + y + "</td><td class='num'>" + fmtMoney(mean(a)) + "</td><td class='num'>" + fmtMoney(median(a)) +
           "</td><td class='num'>" + fmtMoney(Math.min(...a)) + "</td><td class='num'>" + fmtMoney(Math.max(...a)) +
           "</td><td class='num'>" + a.length + "</td></tr>";
  });
  $("summary").innerHTML = table(heads, body);
}

function drawChange(rows) {
  const sec = $("changesec");
  if (!CFG.year_col || !CFG.msrp_col || !CFG.group_col) { sec.classList.add("hidden"); return; }
  const m = new Map(); // group -> year -> values
  const yearSet = new Set();
  rows.forEach(r => {
    const g = r[CFG.group_col], y = r[CFG.year_col], v = r[CFG.msrp_col];
    if (g === null || g === undefined || !isNum(y) || !isNum(v)) return;
    yearSet.add(y);
    if (!m.has(g)) m.set(g, new Map());
    if (!m.get(g).has(y)) m.get(g).set(y, []);
    m.get(g).get(y).push(v);
  });
  const years = [...yearSet].sort((a, b) => a - b);
  if (years.length < 2) {
    Plotly.purge("changechart"); $("changechart").innerHTML = ""; $("changetable").innerHTML = "";
    $("changenote").textContent = years.length
      ? "The data (for this search) only contains model year " + years[0] + ", so price change over time can't be calculated. Add more years to your Excel file to enable this section."
      : "No data for this search.";
    return;
  }
  const y0 = years[0], y1 = years[years.length - 1];
  $("changenote").textContent = "Average starting MSRP per parent company, " + y0 + " → " + y1 + " (only companies with data in both years).";
  const out = [];
  m.forEach((ym, g) => {
    if (ym.has(y0) && ym.has(y1)) {
      const a0 = mean(ym.get(y0)), a1 = mean(ym.get(y1));
      out.push({ g, a0, a1, d: a1 - a0, p: a0 ? (a1 - a0) / a0 * 100 : NaN });
    }
  });
  out.sort((a, b) => b.d - a.d);
  if (!out.length) { $("changechart").innerHTML = ""; $("changetable").innerHTML = "<p class='note' style='padding:10px'>No company has data in both " + y0 + " and " + y1 + ".</p>"; return; }
  const rev = [...out].reverse();
  $("changechart").innerHTML = "";
  Plotly.react("changechart", [{
    type: "bar", orientation: "h", x: rev.map(o => o.d), y: rev.map(o => o.g),
    marker: { color: rev.map(o => o.d >= 0 ? "#2563eb" : "#dc2626") }
  }], {
    title: { text: "Change in average MSRP (" + y0 + " → " + y1 + ")" },
    xaxis: { title: { text: "USD" }, gridcolor: "#e5e7eb" }, margin: { t: 50, l: 220, r: 20, b: 50 },
    height: Math.max(360, out.length * 32 + 120)
  }, { responsive: true });
  const heads = [{t:"Parent Company / Group"},{t:"Avg MSRP " + y0,num:1},{t:"Avg MSRP " + y1,num:1},{t:"Increase (USD)",num:1},{t:"Increase (%)",num:1}];
  $("changetable").innerHTML = table(heads, out.map(o =>
    "<tr><td>" + esc(o.g) + "</td><td class='num'>" + fmtMoney(o.a0) + "</td><td class='num'>" + fmtMoney(o.a1) +
    "</td><td class='num'>" + (o.d >= 0 ? "+" : "-") + fmtMoney(Math.abs(o.d)) + "</td><td class='num'>" +
    (isFinite(o.p) ? o.p.toFixed(1) + "%" : "") + "</td></tr>"));
}

function drawTable(rows) {
  const top = rows.slice(0, 50);
  const thead = "<thead><tr>" + COLS.map(c => "<th>" + esc(c) + "</th>").join("") + "</tr></thead>";
  const tbody = "<tbody>" + top.map(r => "<tr>" + COLS.map(c =>
    "<td class='" + (isNum(r[c]) ? "num" : "") + "'>" + cell(r[c], c) + "</td>").join("") + "</tr>").join("") + "</tbody>";
  $("tbl").innerHTML = thead + tbody;
}

function update() {
  const rows = filtered();
  $("count").textContent = rows.length === DATA.length
    ? "Showing all " + DATA.length.toLocaleString() + " rows"
    : rows.length.toLocaleString() + " of " + DATA.length.toLocaleString() + " rows match your search";
  if (!rows.length) {
    ["scatter","box"].forEach(id => { Plotly.purge(id); });
    $("scatter").innerHTML = "<p class='note'>No data matches your search query.</p>";
    $("box").innerHTML = "";
    $("tbl").innerHTML = ""; $("summary").innerHTML = ""; $("changechart").innerHTML = ""; $("changetable").innerHTML = "";
    return;
  }
  drawScatter(rows); drawBox(rows); drawSummary(rows); drawChange(rows); drawTable(rows);
}

function init() {
  fillSelect($("xsel"), CFG.default_x);
  fillSelect($("ysel"), CFG.default_y);
  let t;
  $("search").addEventListener("input", () => { clearTimeout(t); t = setTimeout(update, 150); });
  $("xsel").addEventListener("change", update);
  $("ysel").addEventListener("change", update);
  $("reset").addEventListener("click", () => {
    $("search").value = "";
    $("xsel").value = CFG.default_x; $("ysel").value = CFG.default_y;
    update();
  });
  update();
}
init();
</script>
</body>
</html>
"""

cols = list(df.columns)
default_x = HP_COL or cols[0]
default_y = MSRP_COL or (cols[-1] if len(cols) > 1 else cols[0])

config = {
    "columns": cols,
    "msrp_col": MSRP_COL,
    "year_col": YEAR_COL,
    "group_col": GROUP_COL,
    "default_x": default_x,
    "default_y": default_y,
}

data_json = df.to_json(orient="records", force_ascii=False)
data_json = data_json.replace("</", "<\\/")   # keep the JSON safe inside <script>
config_json = json.dumps(config, ensure_ascii=False).replace("</", "<\\/")

html = (HTML_TEMPLATE
        .replace("__TITLE__", TITLE)
        .replace("__DATA__", data_json)
        .replace("__CONFIG__", config_json))

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(html)

size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f"✅ Created {OUTPUT_FILE} ({size_kb:,.0f} KB) with {len(df):,} rows.")
print("Next: put it in the root of your GitHub repo and enable Pages (Settings → Pages).")

files.download(OUTPUT_FILE)